In [14]:
import numpy as np
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report, recall_score, confusion_matrix
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
import pickle

class TreeNode():
    def __init__(self, data, feature_idx, feature_val, prediction_probs) -> None:
        self.data = data
        self.feature_idx = feature_idx
        self.feature_val = feature_val
        self.prediction_probs = prediction_probs
        self.left = None
        self.right = None

    def node_def(self) -> str:

        unique_values, value_counts = np.unique(self.data[:,-1], return_counts=True)
        output = ", ".join([f"{value}->{count}" for value, count in zip(unique_values, value_counts)])            
        return f"LEAF | Label Counts = {output} | Pred Probs = {self.prediction_probs}"

class DecisionTreeFromScratch:
    """
    Decision Tree Classifier implemented from scratch
    For training, all possible percentiles of the feature values are used.
    The quality evaluation is done using entropy as a measure.
    """
    def __init__(self, max_depth, min_leaf) -> None:
        """
        Initialize DecisionTree with specified hyperparameters.
        max_depth - maximum depth of the tree.
        min_leaf - minimum number of data points required in a leaf node.
        """
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.tree = None
    
    def _entropy(self, class_probabilities) -> float:
        """Calculates entropy given a list of class probabilities."""
        return sum([-p * np.log2(p) for p in class_probabilities if p > 0])
    
    def _class_probabilities(self, labels) -> list:
        """Returns class probabilities based on label distribution."""
        total_count = len(labels)
        return [count / total_count for count in Counter(labels).values()]
    
    def _data_entropy(self, labels) -> float:
        """Calculates entropy for a given set of labels."""
        return self._entropy(self._class_probabilities(labels))
    
    def _partition_entropy(self, subsets) -> float:
        """Calculates weighted entropy for a given partition of data subsets."""
        total_count = sum(len(subset) for subset in subsets)
        return sum(self._data_entropy(subset) * (len(subset) / total_count) for subset in subsets)
    
    def _split(self, data, feature_index, feature_val) -> tuple:
        """Splits the data into two groups based on a threshold feature value."""
        mask = data[:, feature_index] < feature_val
        group1 = data[mask]
        group2 = data[~mask]
        return group1, group2
    
    def _find_best_split(self, data) -> tuple:
        """
        Finds the best feature and threshold to split data by calculating entropy.
        Returns the split with the lowest entropy.
        """
        min_part_entropy = float('inf')
        best_split = None
        
        n_features = data.shape[1] - 1  # Exclude label column
        for idx in range(n_features):
            feature_vals = np.percentile(data[:, idx], q=np.arange(1, 100, 5))
            for feature_val in feature_vals:
                g1, g2 = self._split(data, idx, feature_val)
                part_entropy = self._partition_entropy([g1[:, -1], g2[:, -1]])
                if part_entropy < min_part_entropy:
                    min_part_entropy = part_entropy
                    best_split = (g1, g2, idx, feature_val, min_part_entropy)
        
        return best_split if best_split else (None, None, None, None, None)
    
    def _find_label_probs(self, data) -> np.array:
        """Calculates label probabilities for a given dataset."""
        labels = data[:, -1].astype(int)
        total_labels = len(labels)
        label_probabilities = np.zeros(len(self.labels_in_train), dtype=float)
        for i, label in enumerate(self.labels_in_train):
            count = np.sum(labels == label)
            label_probabilities[i] = count / total_labels
        return label_probabilities
    
    def _create_tree(self, data, current_depth) -> TreeNode:
        """Recursively builds the decision tree."""
        if current_depth > self.max_depth:
            return None
        
        split = self._find_best_split(data)
        if not split:
            return None
        
        split1, split2, split_feature_idx, split_feature_val, _ = split
        label_probabilities = self._find_label_probs(data)
        
        node = TreeNode(data, split_feature_idx, split_feature_val, label_probabilities)
        
        # Check leaf node condition
        if len(split1) < self.min_leaf or len(split2) < self.min_leaf:
            return node
        
        current_depth += 1
        node.left = self._create_tree(split1, current_depth)
        node.right = self._create_tree(split2, current_depth)
        
        return node
    
    def _predict_one_sample(self, X) -> np.array:
        """Predicts the probability distribution for a single sample."""
        node = self.tree
        while node.left or node.right:
            if X[node.feature_idx] < node.feature_val:
                node = node.left
            else:
                node = node.right
        return node.prediction_probs
    
    def fit(self, X_train, Y_train) -> None:
        """Trains the decision tree model."""
        self.labels_in_train = np.unique(Y_train)
        train_data = np.concatenate((X_train, np.reshape(Y_train, (-1, 1))), axis=1)
        self.tree = self._create_tree(data=train_data, current_depth=0)
    
    def predict_proba(self, X_set) -> np.array:
        """Returns predicted probabilities for each sample in the dataset."""
        return np.apply_along_axis(self._predict_one_sample, 1, X_set)
    
    def predict(self, X_set) -> np.array:
        """Predicts the class labels for a given dataset."""
        pred_probs = self.predict_proba(X_set)
        return np.argmax(pred_probs, axis=1)
    
    def _print_recursive(self, node, level=0) -> None:
        """Recursively prints the tree structure."""
        if node:
            self._print_recursive(node.left, level + 1)
            print('    ' * level + f"-> Feature: {node.feature_idx}, Threshold: {node.feature_val}, Probabilities: {node.prediction_probs}")
            self._print_recursive(node.right, level + 1)
    
    def print_tree(self) -> None:
        """Prints the entire tree structure."""
        self._print_recursive(node=self.tree)
    def evaluate_model(tree, X_val, y_val, labels):
        # Predict the class labels
        predictions = tree.predict(X_val)
        
        # Calculate overall accuracy
        accuracy = accuracy_score(y_val, predictions)
        
        # Calculate recall for each class
        recall_per_class = recall_score(y_val, predictions, average=None, labels=labels)
        
        # Confusion matrix
        conf_matrix = confusion_matrix(y_val, predictions, labels=labels)
        
        # Print metrics
        print("Overall Accuracy:", accuracy)
        print("Recall per Class:")
        for label, recall in zip(labels, recall_per_class):
            print(f"Class {label}: {recall:.2f}")
        print("\nConfusion Matrix:\n", conf_matrix)
        
        return accuracy, recall_per_class, conf_matrix
    
train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply the loaded PCA model to new data
X_new_reduced = pca_loaded.transform(train_data_X)  # X_new is new data



kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []
#for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
#    print(f"Training fold {fold + 1}")
tree = DecisionTreeFromScratch(max_depth=10, min_leaf=4)


# Train the model
tree.fit(X_train=X_train, Y_train=y_train)

# Make predictions
predictions = tree.predict(X_val)
predicted_probabilities = tree.predict_proba(X_val)

#accuracy = accuracy_score(y_val, predictions)

# Output results
print("Predicted labels:", predictions)
print("Predicted probabilities:\n", predicted_probabilities)
#print("Validation Accuracy:", accuracy)
# Optionally, print the tree structure
labels = np.unique(y_train)
accuracy, recall_per_class, conf_matrix = tree.evaluate_model(tree, X_val, y_val, labels)
tree.print_tree()


C:\Users\mihae\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Predicted labels: [3 1 2 ... 4 1 1]
Predicted probabilities:
 [[0.27586207 0.20689655 0.         0.37931034 0.13793103]
 [0.         1.         0.         0.         0.        ]
 [0.         0.00364964 0.91970803 0.         0.07664234]
 ...
 [0.         0.         0.375      0.         0.625     ]
 [0.         1.         0.         0.         0.        ]
 [0.         1.         0.         0.         0.        ]]


TypeError: DecisionTreeFromScratch.evaluate_model() takes 4 positional arguments but 5 were given

In [ ]:
import numpy as np
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
import pickle

train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply the loaded PCA model to new data
X_new_reduced = pca_loaded.transform(train_data_X)  # X_new is new data



C:\Users\mihae\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_new_reduced, train_data_y, test_size=0.2, random_state=42)

# Print shapes to verify the split
print("Training set shape:", X_train.shape, y_train.shape)
print("Validation set shape:", X_val.shape, y_val.shape)

Training set shape: (8000, 50) (8000,)
Validation set shape: (2000, 50) (2000,)


In [12]:

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []
#for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
#    print(f"Training fold {fold + 1}")
tree = DecisionTreeFromScratch(max_depth=10, min_leaf=4)


# Train the model
tree.fit(X_train=X_train, Y_train=y_train)

# Make predictions
predictions = tree.predict(X_val)
predicted_probabilities = tree.predict_proba(X_val)

#accuracy = accuracy_score(y_val, predictions)

# Output results
print("Predicted labels:", predictions)
print("Predicted probabilities:\n", predicted_probabilities)
#print("Validation Accuracy:", accuracy)
# Optionally, print the tree structure
accuracy, recall_per_class, conf_matrix = evaluate_model(tree, X_val, y_val, labels)
tree.print_tree()

Predicted labels: [3 1 2 ... 4 1 1]
Predicted probabilities:
 [[0.27586207 0.20689655 0.         0.37931034 0.13793103]
 [0.         1.         0.         0.         0.        ]
 [0.         0.00364964 0.91970803 0.         0.07664234]
 ...
 [0.         0.         0.375      0.         0.625     ]
 [0.         1.         0.         0.         0.        ]
 [0.         1.         0.         0.         0.        ]]


NameError: name 'evaluate_model' is not defined

In [16]:
import numpy as np
from collections import Counter
from sklearn.metrics import accuracy_score, classification_report, recall_score, confusion_matrix
from sklearn.model_selection import KFold
import pickle

class TreeNode():
    def __init__(self, data, feature_idx, feature_val, prediction_probs) -> None:
        self.data = data
        self.feature_idx = feature_idx
        self.feature_val = feature_val
        self.prediction_probs = prediction_probs
        self.left = None
        self.right = None

    def node_def(self) -> str:
        unique_values, value_counts = np.unique(self.data[:, -1], return_counts=True)
        output = ", ".join([f"{value}->{count}" for value, count in zip(unique_values, value_counts)])            
        return f"LEAF | Label Counts = {output} | Pred Probs = {self.prediction_probs}"

class DecisionTreeFromScratch:
    """
    Decision Tree Classifier implemented from scratch
    For training, all possible percentiles of the feature values are used.
    The quality evaluation is done using entropy as a measure.
    """
    def __init__(self, max_depth, min_leaf) -> None:
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.tree = None
    
    def _entropy(self, class_probabilities) -> float:
        return sum([-p * np.log2(p) for p in class_probabilities if p > 0])
    
    def _class_probabilities(self, labels) -> list:
        total_count = len(labels)
        return [count / total_count for count in Counter(labels).values()]
    
    def _data_entropy(self, labels) -> float:
        return self._entropy(self._class_probabilities(labels))
    
    def _partition_entropy(self, subsets) -> float:
        total_count = sum(len(subset) for subset in subsets)
        return sum(self._data_entropy(subset) * (len(subset) / total_count) for subset in subsets)
    
    def _split(self, data, feature_index, feature_val) -> tuple:
        mask = data[:, feature_index] < feature_val
        group1 = data[mask]
        group2 = data[~mask]
        return group1, group2
    
    def _find_best_split(self, data) -> tuple:
        min_part_entropy = float('inf')
        best_split = None
        n_features = data.shape[1] - 1  # Exclude label column
        for idx in range(n_features):
            feature_vals = np.percentile(data[:, idx], q=np.arange(1, 100, 5))
            for feature_val in feature_vals:
                g1, g2 = self._split(data, idx, feature_val)
                part_entropy = self._partition_entropy([g1[:, -1], g2[:, -1]])
                if part_entropy < min_part_entropy:
                    min_part_entropy = part_entropy
                    best_split = (g1, g2, idx, feature_val, min_part_entropy)
        return best_split if best_split else (None, None, None, None, None)
    
    def _find_label_probs(self, data) -> np.array:
        labels = data[:, -1].astype(int)
        total_labels = len(labels)
        label_probabilities = np.zeros(len(self.labels_in_train), dtype=float)
        for i, label in enumerate(self.labels_in_train):
            count = np.sum(labels == label)
            label_probabilities[i] = count / total_labels
        return label_probabilities
    
    def _create_tree(self, data, current_depth) -> TreeNode:
        if current_depth > self.max_depth:
            return None
        
        split = self._find_best_split(data)
        if not split:
            return None
        
        split1, split2, split_feature_idx, split_feature_val, _ = split
        label_probabilities = self._find_label_probs(data)
        
        node = TreeNode(data, split_feature_idx, split_feature_val, label_probabilities)
        if len(split1) < self.min_leaf or len(split2) < self.min_leaf:
            return node
        
        current_depth += 1
        node.left = self._create_tree(split1, current_depth)
        node.right = self._create_tree(split2, current_depth)
        
        return node
    
    def _predict_one_sample(self, X) -> np.array:
        node = self.tree
        while node.left or node.right:
            if X[node.feature_idx] < node.feature_val:
                node = node.left
            else:
                node = node.right
        return node.prediction_probs
    
    def fit(self, X_train, Y_train) -> None:
        self.labels_in_train = np.unique(Y_train)
        train_data = np.concatenate((X_train, np.reshape(Y_train, (-1, 1))), axis=1)
        self.tree = self._create_tree(data=train_data, current_depth=0)
    
    def predict_proba(self, X_set) -> np.array:
        return np.apply_along_axis(self._predict_one_sample, 1, X_set)
    
    def predict(self, X_set) -> np.array:
        pred_probs = self.predict_proba(X_set)
        return np.argmax(pred_probs, axis=1)
    
    def _print_recursive(self, node, level=0) -> None:
        if node:
            self._print_recursive(node.left, level + 1)
            print('    ' * level + f"-> Feature: {node.feature_idx}, Threshold: {node.feature_val}, Probabilities: {node.prediction_probs}")
            self._print_recursive(node.right, level + 1)
    
    def print_tree(self) -> None:
        self._print_recursive(node=self.tree)
    
    def evaluate_model(self, X_val, y_val, labels):
        predictions = self.predict(X_val)
        accuracy = accuracy_score(y_val, predictions)
        recall_per_class = recall_score(y_val, predictions, average=None, labels=labels)
        conf_matrix = confusion_matrix(y_val, predictions, labels=labels)
        
        print("Overall Accuracy:", accuracy)
        print("Recall per Class:")
        for label, recall in zip(labels, recall_per_class):
            print(f"Class {label}: {recall:.2f}")
        print("\nConfusion Matrix:\n", conf_matrix)
        
        return accuracy, recall_per_class, conf_matrix

# Load data
train_data = np.load('Data+Description/fashion_train.npy')  
train_data_X = train_data[:, :-1]
train_data_y = train_data[:, -1]
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply PCA
X_new_reduced = pca_loaded.transform(train_data_X)

# K-Fold Evaluation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
fold_recalls = []
fold_conf_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"\nTraining Fold {fold + 1}")
    
    X_train, X_val = X_new_reduced[train_index], X_new_reduced[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    tree = DecisionTreeFromScratch(max_depth=10, min_leaf=4)
    tree.fit(X_train, y_train)
    labels = np.unique(y_train)
    
    accuracy, recall_per_class, conf_matrix = tree.evaluate_model(X_val, y_val, labels)
    fold_accuracies.append(accuracy)
    fold_recalls.append(recall_per_class)
    fold_conf_matrices.append(conf_matrix)
    print(f"Fold {fold + 1} Accuracy: {accuracy:.2f}")

print("\nCross-Validation Results")
print(f"Mean Accuracy: {np.mean(fold_accuracies):.2f}")
print("Mean Recall per Class:", np.mean(fold_recalls, axis=0))


C:\Users\mihae\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Training Fold 1
Overall Accuracy: 0.7765
Recall per Class:
Class 0: 0.77
Class 1: 0.90
Class 2: 0.79
Class 3: 0.87
Class 4: 0.57

Confusion Matrix:
 [[321   2  16  27  49]
 [  2 341   6  24   6]
 [  5   2 306   9  66]
 [ 16  10  12 351  16]
 [ 75   6  70  28 234]]
Fold 1 Accuracy: 0.78

Training Fold 2
Overall Accuracy: 0.7535
Recall per Class:
Class 0: 0.73
Class 1: 0.94
Class 2: 0.74
Class 3: 0.82
Class 4: 0.54

Confusion Matrix:
 [[289   4  10  33  62]
 [  3 371   5  13   1]
 [ 18   0 313  14  80]
 [ 23  13  10 323  27]
 [ 85   4  61  27 211]]
Fold 2 Accuracy: 0.75

Training Fold 3
Overall Accuracy: 0.7385
Recall per Class:
Class 0: 0.68
Class 1: 0.93
Class 2: 0.81
Class 3: 0.81
Class 4: 0.49

Confusion Matrix:
 [[273   6  19  23  78]
 [  7 371   4  13   3]
 [  9   1 296   5  55]
 [ 29  16  11 322  20]
 [ 92   3 102  27 215]]
Fold 3 Accuracy: 0.74

Training Fold 4
Overall Accuracy: 0.7755
Recall per Class:
Class 0: 0.71
Class 1: 0.92
Class 2: 0.81
Class 3: 0.83
Class 4: 0.61

Confu

In [ ]:
#Best working version so far - 1.20 minutes, normal accuracy and recall

import numpy as np
from collections import Counter
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from sklearn.model_selection import KFold
import pickle

class TreeNode:
    def __init__(self, data, feature_idx, feature_val, prediction_probs):
        self.data = data
        self.feature_idx = feature_idx
        self.feature_val = feature_val
        self.prediction_probs = prediction_probs
        self.left = None
        self.right = None

    def node_def(self):
        unique_values, value_counts = np.unique(self.data[:, -1], return_counts=True)
        output = ", ".join([f"{value}->{count}" for value, count in zip(unique_values, value_counts)])
        return f"LEAF | Label Counts = {output} | Pred Probs = {self.prediction_probs}"

class DecisionTreeFromScratch:
    def __init__(self, max_depth, min_leaf):
        self.max_depth = max_depth
        self.min_leaf = min_leaf
        self.tree = None

    def _entropy(self, class_probabilities):
        return sum([-p * np.log2(p) for p in class_probabilities if p > 0])

    def _class_probabilities(self, labels):
        total_count = len(labels)
        return [count / total_count for count in Counter(labels).values()]

    def _data_entropy(self, labels):
        return self._entropy(self._class_probabilities(labels))

    def _partition_entropy(self, subsets):
        total_count = sum(len(subset) for subset in subsets)
        return sum(self._data_entropy(subset) * (len(subset) / total_count) for subset in subsets)

    def _split_indices(self, data, feature_index, feature_val):
        mask = data[:, feature_index] < feature_val
        return np.where(mask)[0], np.where(~mask)[0]

    def _find_best_split(self, data):
        min_part_entropy = float('inf')
        best_split = None
        n_features = data.shape[1] - 1  # Exclude label column

        for idx in range(n_features):
            feature_vals = np.percentile(data[:, idx], q=np.linspace(0, 100, num=20))  # Optimize thresholds
            for feature_val in feature_vals:
                left_idx, right_idx = self._split_indices(data, idx, feature_val)
                if len(left_idx) == 0 or len(right_idx) == 0:  # Skip invalid splits
                    continue
                part_entropy = self._partition_entropy([data[left_idx, -1], data[right_idx, -1]])
                if part_entropy < min_part_entropy:
                    min_part_entropy = part_entropy
                    best_split = (left_idx, right_idx, idx, feature_val, min_part_entropy)

        return best_split if best_split else (None, None, None, None, None)

    def _find_label_probs(self, labels):
        total_labels = len(labels)
        label_probabilities = np.zeros(len(self.labels_in_train), dtype=float)
        for i, label in enumerate(self.labels_in_train):
            count = np.sum(labels == label)
            label_probabilities[i] = count / total_labels
        return label_probabilities

    def _create_tree(self, data, indices, current_depth):
        if current_depth >= self.max_depth or len(indices) < self.min_leaf:
            return None

        split = self._find_best_split(data[indices])
        if not split:
            return None

        left_idx, right_idx, split_feature_idx, split_feature_val, _ = split
        left_data_indices = indices[left_idx]
        right_data_indices = indices[right_idx]
        label_probabilities = self._find_label_probs(data[indices, -1])

        node = TreeNode(data[indices], split_feature_idx, split_feature_val, label_probabilities)
        if len(left_data_indices) < self.min_leaf or len(right_data_indices) < self.min_leaf:
            return node

        current_depth += 1
        node.left = self._create_tree(data, left_data_indices, current_depth)
        node.right = self._create_tree(data, right_data_indices, current_depth)
        return node

    def _predict_one_sample(self, X):
        node = self.tree
        while node.left or node.right:
            if X[node.feature_idx] < node.feature_val:
                node = node.left
            else:
                node = node.right
        return node.prediction_probs

    def fit(self, X_train, Y_train):
        self.labels_in_train = np.unique(Y_train)
        train_data = np.concatenate((X_train, np.reshape(Y_train, (-1, 1))), axis=1)
        indices = np.arange(len(train_data))
        self.tree = self._create_tree(data=train_data, indices=indices, current_depth=0)

    def predict_proba(self, X_set):
        return np.apply_along_axis(self._predict_one_sample, 1, X_set)

    def predict(self, X_set):
        pred_probs = self.predict_proba(X_set)
        return np.argmax(pred_probs, axis=1)

    def _print_recursive(self, node, level=0):
        if node:
            self._print_recursive(node.left, level + 1)
            print('    ' * level + f"-> Feature: {node.feature_idx}, Threshold: {node.feature_val}, Probabilities: {node.prediction_probs}")
            self._print_recursive(node.right, level + 1)

    def print_tree(self):
        self._print_recursive(node=self.tree)

    def evaluate_model(self, X_val, y_val, labels):
        predictions = self.predict(X_val)
        accuracy = accuracy_score(y_val, predictions)
        recall_per_class = recall_score(y_val, predictions, average=None, labels=labels)
        conf_matrix = confusion_matrix(y_val, predictions, labels=labels)

        print("Overall Accuracy:", accuracy)
        print("Recall per Class:")
        for label, recall in zip(labels, recall_per_class):
            print(f"Class {label}: {recall:.2f}")
        print("\nConfusion Matrix:\n", conf_matrix)

        return accuracy, recall_per_class, conf_matrix

# Main Code: Data Loading and Cross-Validation
train_data = np.load('Data+Description/fashion_train.npy')
train_data_X = train_data[:, :-1]
train_data_y = train_data[:, -1]

with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

X_new_reduced = pca_loaded.transform(train_data_X)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
fold_recalls = []
fold_conf_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"\nTraining Fold {fold + 1}")

    X_train, X_val = X_new_reduced[train_index], X_new_reduced[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]

    tree = DecisionTreeFromScratch(max_depth=10, min_leaf=4)
    tree.fit(X_train, y_train)
    labels = np.unique(y_train)

    accuracy, recall_per_class, conf_matrix = tree.evaluate_model(X_val, y_val, labels)
    fold_accuracies.append(accuracy)
    fold_recalls.append(recall_per_class)
    fold_conf_matrices.append(conf_matrix)
    print(f"Fold {fold + 1} Accuracy: {accuracy:.2f}")

print("\nCross-Validation Results")
print(f"Mean Accuracy: {np.mean(fold_accuracies):.2f}")
print("Mean Recall per Class:", np.mean(fold_recalls, axis=0))


C:\Users\mihae\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(



Training Fold 1
Overall Accuracy: 0.7635
Recall per Class:
Class 0: 0.73
Class 1: 0.91
Class 2: 0.81
Class 3: 0.80
Class 4: 0.59

Confusion Matrix:
 [[301   7   9  18  80]
 [  6 345   3  23   2]
 [  9   0 313  13  53]
 [ 38   9  10 326  22]
 [ 64   5  80  22 242]]
Fold 1 Accuracy: 0.76

Training Fold 2
Overall Accuracy: 0.7575
Recall per Class:
Class 0: 0.67
Class 1: 0.94
Class 2: 0.81
Class 3: 0.83
Class 4: 0.53

Confusion Matrix:
 [[268   2   9  44  75]
 [  2 370   4  14   3]
 [  7   1 344  22  51]
 [ 25  16   9 327  19]
 [ 68   3  81  30 206]]
Fold 2 Accuracy: 0.76

Training Fold 3
Overall Accuracy: 0.7645
Recall per Class:
Class 0: 0.72
Class 1: 0.93
Class 2: 0.83
Class 3: 0.87
Class 4: 0.50

Confusion Matrix:
 [[289   8   9  34  59]
 [  2 372   5  18   1]
 [  7   1 303  13  42]
 [ 23   7   8 346  14]
 [ 94   1  84  41 219]]
Fold 3 Accuracy: 0.76

Training Fold 4
Overall Accuracy: 0.774
Recall per Class:
Class 0: 0.71
Class 1: 0.90
Class 2: 0.79
Class 3: 0.87
Class 4: 0.58

Confus